# Xarray-Spatial Performance: Rechunking Without Shuffle

When working with large dask-backed rasters, rechunking to bigger blocks can
speed up downstream operations like `slope()` or `focal_mean()` that use
`map_overlap`. But if the new chunk size is not an exact multiple of the
original, dask has to split and recombine blocks, which tanks performance.
`rechunk_no_shuffle` picks the largest whole-chunk multiple that fits your
target size, so dask can merge blocks in place with zero shuffle overhead.

### What you'll build

1. Create a synthetic dask-backed raster with small chunks
2. Rechunk to a ~64 MB target without triggering a shuffle
3. Use the `.xrs` accessor shortcut
4. Compare task graph sizes before and after rechunking
5. Confirm numpy arrays pass through unchanged

[Data](#Data) · [Rechunk to ~64-MB target](#Rechunk-to-~64-MB-target) · [.xrs accessor](#Using-the-.xrs-accessor) · [Task graph comparison](#Compare-task-graph-sizes) · [Numpy passthrough](#Non-dask-arrays-pass-through-unchanged)

Standard imports plus the rechunk utility.

In [ ]:
import numpy as np
import dask.array as da
import xarray as xr
import matplotlib.pyplot as plt

import xrspatial
from xrspatial.utils import rechunk_no_shuffle

## Data

Start with a 4096 x 4096 float32 raster chunked at 256 x 256 (about 0.25 MB
per chunk).

In [ ]:
np.random.seed(42)
raw = np.random.rand(4096, 4096).astype(np.float32) * 1000
dem = xr.DataArray(
    da.from_array(raw, chunks=256),
    dims=['y', 'x'],
    coords={
        'y': np.linspace(40.0, 41.0, 4096),
        'x': np.linspace(-105.0, -104.0, 4096),
    },
)
print(f'Original chunks: {dem.chunks}')
print(f'Chunks per axis:  {len(dem.chunks[0])} x {len(dem.chunks[1])}')

A random synthetic elevation surface, chunked into a 16 x 16 grid of 256-pixel blocks.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
dem.plot.imshow(ax=ax, cmap='terrain', add_colorbar=True)
ax.set_title('Synthetic DEM (random, 4096x4096)')
plt.tight_layout()
plt.show()

## Rechunk to ~64 MB target

Each new chunk will be an exact multiple of 256, so dask just groups
existing blocks together with no data movement.

In [ ]:
big = rechunk_no_shuffle(dem, target_mb=64)
print(f'New chunks:      {big.chunks}')
print(f'Chunks per axis: {len(big.chunks[0])} x {len(big.chunks[1])}')
print(f'Block size:      {big.chunks[0][0]} x {big.chunks[1][0]}')
print(f'Multiple of 256: {big.chunks[0][0] // 256}x')

## Using the .xrs accessor

The same function is available directly on any DataArray via the `.xrs` accessor.

In [ ]:
big_via_accessor = dem.xrs.rechunk_no_shuffle(target_mb=64)
print(f'Accessor chunks: {big_via_accessor.chunks}')
assert big.chunks == big_via_accessor.chunks

## Compare task graph sizes

Fewer, larger chunks means a smaller task graph for downstream operations.
Here we compare the graph size of `slope()` on the original vs rechunked raster.

In [ ]:
from xrspatial.slope import slope

slope_small = slope(dem)
slope_big   = slope(big)

print(f'slope() graph with original chunks: {len(dict(slope_small.data.__dask_graph__())):,} tasks')
print(f'slope() graph with rechunked:       {len(dict(slope_big.data.__dask_graph__())):,} tasks')

## Non-dask arrays pass through unchanged

If the input is a plain numpy-backed DataArray, the function returns it
as-is. No copy, no error.

In [ ]:
numpy_dem = xr.DataArray(raw, dims=['y', 'x'])
result = rechunk_no_shuffle(numpy_dem, target_mb=64)
assert result is numpy_dem
print('Numpy passthrough: OK')

### References

- [Dask Array Chunks](https://docs.dask.org/en/stable/array-chunks.html), Dask documentation
- [Best Practices for Rechunking](https://docs.dask.org/en/stable/array-best-practices.html#select-a-good-chunk-size), Dask documentation
- [xarray.DataArray.chunk](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.chunk.html), xarray documentation